In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:

def process_all_pdfs(files):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(r"C:\Users\ASUS\OneDrive\Desktop\Project\PDFs")
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Adding information
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs
all_pdf_documents = process_all_pdfs("../data")

invalid pdf header: b' %PDF'


Found 2 PDF files to process

Processing: Generative_AI_Notes_Expanded.pdf
Loaded 6 pages

Processing: Python_Notes.pdf
Loaded 37 pages

Total documents loaded: 43


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T07:22:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T07:22:36+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\ASUS\\OneDrive\\Desktop\\Project\\PDFs\\Generative_AI_Notes_Expanded.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Generative_AI_Notes_Expanded.pdf', 'file_type': 'pdf'}, page_content='Generative AI – Comprehensive Course Notes\n1. Introduction to Generative AI\nGenerative Artificial Intelligence refers to models capable of producing new, high-quality data such\nas images, text, audio, and designs. Unlike traditional AI systems that classify or predict,\nGenerative AI creates original outputs using learned patterns in data.\nDiscriminative vs Generative Models:\n\x7f Discriminative Models: Learn P(y|x). Example: Logistic Regression, SVM.\n\x7

In [ ]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents in 1000 chunks size for better performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

Split 43 documents into 47 chunks

Example chunk:
Content: Generative AI – Comprehensive Course Notes
1. Introduction to Generative AI
Generative Artificial Intelligence refers to models capable of producing new, high-quality data such
as images, text, audio,...
Metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T07:22:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T07:22:36+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\ASUS\\OneDrive\\Desktop\\Project\\PDFs\\Generative_AI_Notes_Expanded.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Generative_AI_Notes_Expanded.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T07:22:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T07:22:36+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\ASUS\\OneDrive\\Desktop\\Project\\PDFs\\Generative_AI_Notes_Expanded.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Generative_AI_Notes_Expanded.pdf', 'file_type': 'pdf'}, page_content='Generative AI – Comprehensive Course Notes\n1. Introduction to Generative AI\nGenerative Artificial Intelligence refers to models capable of producing new, high-quality data such\nas images, text, audio, and designs. Unlike traditional AI systems that classify or predict,\nGenerative AI creates original outputs using learned patterns in data.\nDiscriminative vs Generative Models:\n\x7f Discriminative Models: Learn P(y|x). Example: Logistic Regression, SVM.\n\x7

In [ ]:
import numpy as np
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings

import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from typing import List
import numpy as np
from langchain_community.embeddings import OllamaEmbeddings


class EmbeddingManager:
    """Handles document embedding generation using Ollama embeddings"""

    def __init__(self, model_name: str = "nomic-embed-text", base_url: str = "http://localhost:11434"):
        """
        Initialize the embedding manager

        Args:
            model_name: Ollama embedding model name
            base_url: Ollama server URL
        """
        self.model_name = model_name
        self.base_url = base_url
        self.model = None
        self.embedding_dim = None
        self._load_model()

    def _load_model(self):
        """Initialize Ollama embedding model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = OllamaEmbeddings(
                model=self.model_name,
                base_url=self.base_url
            )

            # Infer embedding dimension with a dummy call
            test_embedding = self.model.embed_query("test")
            self.embedding_dim = len(test_embedding)

            print(f"Model loaded successfully. Embedding dimension: {self.embedding_dim}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        if not texts:
            return np.empty((0, self.embedding_dim))

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.embed_documents(texts)

        embeddings = np.array(embeddings, dtype=np.float32)

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()


Loading embedding model: nomic-embed-text
Model loaded successfully. Embedding dimension: 768


In [ ]:
import os
import uuid
from typing import List, Any
import numpy as np

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.embeddings import OllamaEmbeddings


class VectorStore:
    """Manages document embeddings in a FAISS vector store (Chroma-like API)"""

    def __init__(
        self,
        index_path: str = "../data/faiss_index",
        model_name: str = "nomic-embed-text",
        base_url: str = "http://localhost:11434",
    ):
        """
        Initialize the FAISS vector store

        Args:
            index_path: Directory to persist FAISS index
            model_name: Ollama embedding model name
            base_url: Ollama server URL
        """
        self.index_path = index_path
        self.model_name = model_name
        self.base_url = base_url

        self.embeddings = OllamaEmbeddings(
            model=self.model_name,
            base_url=self.base_url,
        )

        self.vectorstore = None
        self._initialize_store()

    def _initialize_store(self):
        """Load or create FAISS index"""
        try:
            if os.path.exists(self.index_path):
                self.vectorstore = FAISS.load_local(
                    self.index_path,
                    self.embeddings,
                    allow_dangerous_deserialization=True,
                )
                print("Vector store initialized.")
                print(f"Existing documents in store: {self.vectorstore.index.ntotal}")
            else:
                os.makedirs(self.index_path, exist_ok=True)
                print("No existing FAISS index found.")
                print("A new FAISS vector store will be created.")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the FAISS vector store

        Args:
            documents: List of LangChain Document objects
            embeddings: NumPy array with shape (n_docs, embedding_dim)
        """
        if not documents:
            print("No documents to add")
            return

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data (Chroma-like explicit handling)
        texts = []
        metadatas = []
        ids = []

        for i, doc in enumerate(documents):
            texts.append(doc.page_content)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")

        embeddings = np.array(embeddings, dtype=np.float32)

        # Create or extend FAISS store
        if self.vectorstore is None:
            self.vectorstore = FAISS.from_embeddings(
                text_embeddings=list(zip(texts, embeddings)),
                embedding=self.embeddings,
                metadatas=metadatas,
                ids=ids,
            )
        else:
            self.vectorstore.add_embeddings(
                text_embeddings=list(zip(texts, embeddings)),
                metadatas=metadatas,
                ids=ids,
            )

        self.vectorstore.save_local(self.index_path)

        print(f"Successfully added {len(documents)} documents to vector store")
        print(f"Total documents in store: {self.vectorstore.index.ntotal}")


In [ ]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T07:22:36+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T07:22:36+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\ASUS\\OneDrive\\Desktop\\Project\\PDFs\\Generative_AI_Notes_Expanded.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'Generative_AI_Notes_Expanded.pdf', 'file_type': 'pdf'}, page_content='Generative AI – Comprehensive Course Notes\n1. Introduction to Generative AI\nGenerative Artificial Intelligence refers to models capable of producing new, high-quality data such\nas images, text, audio, and designs. Unlike traditional AI systems that classify or predict,\nGenerative AI creates original outputs using learned patterns in data.\nDiscriminative vs Generative Models:\n\x7f Discriminative Models: Learn P(y|x). Example: Logistic Regression, SVM.\n\x7

In [ ]:
import os
import uuid
from typing import List
import numpy as np

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

from langchain_community.embeddings import OllamaEmbeddings


class VectorStore:
    """Manages document embeddings in a FAISS vector store"""

    def __init__(
        self,
        index_path: str = "../data/faiss_index",
        model_name: str = "nomic-embed-text",
        base_url: str = "http://localhost:11434",
    ):
        """
        Initialize the FAISS vector store
        """
        self.index_path = index_path

        # Embedding model (Ollama)
        self.embeddings = OllamaEmbeddings(
            model=model_name,
            base_url=base_url,
        )

        self.vectorstore = None
        self._initialize_store()

    def _initialize_store(self):
        """Load or initialize FAISS index"""
        if os.path.exists(self.index_path):
            self.vectorstore = FAISS.load_local(
                self.index_path,
                self.embeddings,
                allow_dangerous_deserialization=True,
            )
            print("Vector store initialized.")
            print(f"Existing documents in store: {self.vectorstore.index.ntotal}")
        else:
            os.makedirs(self.index_path, exist_ok=True)
            print("No existing FAISS index found.")
            print("A new FAISS vector store will be created.")

    def add_documents(self, documents: List[Document]):
        """
        Add documents to the FAISS vector store
        (Embeddings are generated internally)
        """
        if not documents:
            print("No documents to add")
            return

        print(f"Adding {len(documents)} documents to vector store...")

        texts = [doc.page_content for doc in documents]
        metadatas = []
        ids = []

        for i, doc in enumerate(documents):
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")

        # Create or update FAISS index
        if self.vectorstore is None:
            self.vectorstore = FAISS.from_texts(
                texts=texts,
                embedding=self.embeddings,
                metadatas=metadatas,
                ids=ids,
            )
        else:
            self.vectorstore.add_texts(
                texts=texts,
                metadatas=metadatas,
                ids=ids,
            )

        self.vectorstore.save_local(self.index_path)

        print(f"Successfully added {len(documents)} documents")
        print(f"Total documents in store: {self.vectorstore.index.ntotal}")

vectorstore = VectorStore()
vectorstore.add_documents(chunks)


Vector store initialized.
Existing documents in store: 188
Adding 47 documents to vector store...
Successfully added 47 documents
Total documents in store: 235


In [ ]:
from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from a FAISS vector store"""

    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0,
    ) -> List[Dict[str, Any]]:

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.vectorstore.similarity_search_with_score_by_vector(
                embedding=query_embedding,
                k=top_k,
            )

            retrieved_docs = []

            if not results:
                print("No documents found")
                return []

            for rank, (doc, distance) in enumerate(results, start=1):
                similarity_score = 1 / (1 + distance)

                if similarity_score < score_threshold:
                    continue

                retrieved_docs.append(
                    {
                        "id": doc.metadata.get("id", f"rank_{rank}"),
                        "content": doc.page_content,
                        "metadata": doc.metadata,
                        "similarity_score": similarity_score,
                        "distance": distance,
                        "rank": rank,
                    }
                )

            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


In [ ]:
vectorstore = VectorStore()
embedding_manager = EmbeddingManager()

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

results = rag_retriever.retrieve("What are data types in python?")
results

Vector store initialized.
Existing documents in store: 235
Loading embedding model: nomic-embed-text
Model loaded successfully. Embedding dimension: 768
Retrieving documents for query: 'What are data types in python?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 768)
Retrieved 5 documents (after filtering)


[{'id': 'rank_1',
  'content': 'Ranjith Kumar. M  CSE, SRITW \n6 Chapter 2: Data Types  \nMemory Operations  \n\uf0b7 No declaration  \n\uf0b7 Store values in variable name  \n\uf0b7 Retrieve value  \n\uf0b7 Delete variable (del)  \n\uf0b7 Constants / arithmetic operations decide variable types  \n \nExample:  \n>>> 10 \n>>> a = 10  \n>>> del(a)  \n \nTypes:  \n\uf0b7 Int \n\uf0b7 Float  \n\uf0b7 String  \n\uf0b7 Boolean  \n \nNumbers:  \n- int (10)  \n - long (0122L) - in python3 no long datatype  \n - float (15.20)  \n - Complex (3.4j) - j might be lower case or upper case  \n - Type conversion  \n  - int() \n  - float()',
  'metadata': {'producer': 'doPDF Ver 9.0 Build 219',
   'creator': 'PyPDF',
   'creationdate': '2019-09-07T21:34:12+05:30',
   'source': 'C:\\Users\\ASUS\\OneDrive\\Desktop\\Project\\PDFs\\Python_Notes.pdf',
   'total_pages': 37,
   'page': 5,
   'page_label': '6',
   'source_file': 'Python_Notes.pdf',
   'file_type': 'pdf'},
  'similarity_score': 0.00567814969396

In [ ]:
rag_retriever.retrieve("field where llm can be used")

Retrieving documents for query: 'field where llm can be used'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 768)
Retrieved 5 documents (after filtering)


[{'id': 'rank_1',
  'content': '6. Applications, Trends, and Ethics\nGenerative AI is transforming fields:\n\x7f Healthcare (drug discovery, imaging)\n\x7f Art, design, and entertainment\n\x7f Education and personalized learning\n\x7f Music and content creation\nEthical Concerns:\n\x7f Deepfakes and misinformation\n\x7f Copyright violations\n\x7f Bias and fairness\n\x7f Data privacy and regulation\nFuture Trends:\n\x7f Diffusion models (state-of-the-art image generation)\n\x7f Multimodal models (text-image-audio integration)\n\x7f Real-time generative applications\n\x7f AI-driven scientific discovery',
  'metadata': {'producer': 'ReportLab PDF Library - www.reportlab.com',
   'creator': '(unspecified)',
   'creationdate': '2025-11-20T07:22:36+00:00',
   'author': '(anonymous)',
   'keywords': '',
   'moddate': '2025-11-20T07:22:36+00:00',
   'subject': '(unspecified)',
   'title': '(anonymous)',
   'trapped': '/False',
   'source': 'C:\\Users\\ASUS\\OneDrive\\Desktop\\Project\\PDFs\\Ge

In [ ]:
#import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
groq_api = os.getenv("GROQ_API_KEY")

#from langchain.prompts.prompt import PromptTemplate
#from langchain.schema import HumanMessage, SystemMessage

llm = ChatGroq(groq_api_key=groq_api,model_name="llama-3.1-8b-instant", temperature =0.1,max_tokens=1024)
def  rag(query,retriever,llm,top_k=3):
    result = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content']for doc in results]) if results else ""
    if not context:
        return "No relevant context."
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}
        Question: {query}
        Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [ ]:
answer = rag("Python supports what data structures?",rag_retriever,llm)
answer

Retrieving documents for query: 'Python supports what data structures?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 768)
Retrieved 3 documents (after filtering)


'Python supports the following data structures:\n\n1. Integers (int)\n2. Floating point numbers (float)\n3. Complex numbers (Complex)\n4. Strings\n5. Boolean values'

In [ ]:
answer = rag("how two digits numebers there are in number system?",rag_retriever,llm)
answer

Retrieving documents for query: 'how two digits numebers there are in number system?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 768)
Retrieved 3 documents (after filtering)


'There is no information in the provided context about the number system or the concept of two-digit numbers. The context is about Python data types and memory operations.'